# Install Packages

In [ ]:
# pip install git+https://github.com/dnth/rag-datakit.git
# !pip install tiktoken

In [ ]:
# !pip install ipywidgets
# !pip install python-dotenv


# Load ENV

In [1]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get token from environment
token = os.getenv("HF_TOKEN")
login(token=token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Load SSF Data

In [2]:
from datasets import load_dataset

dataset = load_dataset("dnth/ssf-dataset")
dataset

DatasetDict({
    train: Dataset({
        features: ['Sector', 'Track', 'Job Role', 'Job Role Description', 'Performance Expectation'],
        num_rows: 1885
    })
})

In [3]:
dataset["train"][0]

{'Sector': 'Accountancy',
 'Track': 'Assurance',
 'Job Role': 'Audit Associate / Audit Assistant Associate',
 'Job Role Description': 'The Audit Associate/Audit Assistant Associate undertakes specific stages of audit work under supervision. He/She begins to appreciate the underlying principles behind the tasks assigned to him as part of the audit plan. He is also able to make adjustments to the application of skills to improve the work tasks or solve non-complex issues. The Audit Associate/Audit Assistant Associate operates in a structured work environment. He is able to build relationships, work in a team and identify ethical issues with reference to the code of professional conduct and ethics. He is able to select and apply from a range of known solutions to familiar problems and takes responsibility for his own learning and performance. He is a trustworthy and meticulous individual.',
 'Performance Expectation': 'In accordance with: Singapore Standards on Auditing, Ethics Pronouncem

# SSF Job Description Token Analysis

In [4]:
import tiktoken
from datasets import load_dataset
import numpy as np
import math

# Load the OpenAI API key from environment

text_column = 'Job Role Description'

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Get the Job Role Description column
job_descriptions = dataset['train'][text_column]

# Calculate token lengths for all job descriptions
token_lengths = [num_tokens_from_string(description, "cl100k_base") for description in job_descriptions]

# Calculate the average, max, min, and other statistics
average_token_length = np.mean(token_lengths)
max_token_length = np.max(token_lengths)
min_token_length = np.min(token_lengths)
std_dev_token_length = np.std(token_lengths)  # Standard deviation

# Calculate percentiles
p25 = np.percentile(token_lengths, 25)  # 25th percentile
p50 = np.percentile(token_lengths, 50)  # 50th percentile (median)
p75 = np.percentile(token_lengths, 75)  # 75th percentile

# Calculate IQR (Interquartile Range)
IQR = p75 - p25
lower_bound = p25 - 1.5 * IQR
upper_bound = p75 + 1.5 * IQR

# Detect outliers
outliers = [length for length in token_lengths if length < lower_bound or length > upper_bound]

# Print the exact average token length
print(f"Exact average token length: {average_token_length}")

# Round up the average token length to the nearest integer
token_avg_length_rounded = math.ceil(average_token_length)

# Print the rounded-up average token length
print(f"Rounded up average token length: ~{token_avg_length_rounded}")

# Print the max and min token lengths
print(f"Maximum token length: {max_token_length}")
print(f"Minimum token length: {min_token_length}")

# Print the standard deviation and variance
print(f"Standard deviation: {std_dev_token_length}")

# Print percentiles
print(f"25th Percentile: {p25}")
print(f"50th Percentile (Median): {p50}")
print(f"75th Percentile: {p75}")

# Print outliers
print(f"Number of outliers: {len(outliers)}")



Exact average token length: 161.60371352785145
Rounded up average token length: ~162
Maximum token length: 393
Minimum token length: 52
Standard deviation: 49.290052444984674
25th Percentile: 124.0
50th Percentile (Median): 155.0
75th Percentile: 195.0
Number of outliers: 14


# Synthetic Data Generation Setup

In [34]:
import os
from distilabel.models import OpenAILLM, TransformersLLM

# llm = TransformersLLM(
#     model="Qwen/Qwen3-4B-Instruct-2507",
#     device_map="auto",
#     torch_dtype="float16",
# )

llm = OpenAILLM(
    model="gpt-4o-mini",
    # model="gpt-5-mini-2025-08-07",
    api_key=os.getenv("OPENAI_API_KEY"),
)


In [ ]:
context = """
You are an HR assistant tasked with generating realistic job descriptions based on a Singapore SkillsFuture Framework input. 
For each job, you will create **one positive description** and **one negative description**, randomly selecting the negative type from the five strategies below.

### Input:
A job description containing:
- Job title (e.g., Audit Associate)
- Role responsibilities and duties
- Work environment and supervision structure
- Required skills and attributes
- Professional conduct expectations

### Output Instructions:

#### 1. Positive Description
- Start with "The [Job Role]"
- Capture the essence of the original role using different words
- Keep the same seniority level and core responsibilities
- Use varied terminology naturally
- Include specific responsibilities, skills, and requirements
- Read as if a different organization is posting a similar role

#### 2. Negative Description
- Randomly select **one** strategy from the five below for each job
- Start with "The [Job Role]" 
- Don't label the negative type
- Include some similar keywords but **change the intent, context, or responsibilities**

**Negative Strategies**:

1. **Easy Negative - Different Function, Same Industry**
   - Change the core function but keep the same industry
   - Use completely different skills
   - Maintain professional context
   - Example: Audit Associate → Tax Associate

2. **Medium Negative - Same Industry, Different Seniority**
   - Change responsibility level (Junior ↔ Senior)
   - Alter supervision structure
   - Modify years of experience or decision-making authority
   - Example: Audit Associate → Senior Audit Manager

3. **Hard Negative - Same Skills, Different Domain**
   - Transfer core skills to a different industry
   - Maintain similar analytical/technical requirements
   - Change regulatory environment or business context
   - Example: Audit Associate → Compliance Associate (Banking)

4. **Hard Negative - Geographic/Regulatory Variation**
   - Same role but different regulatory or geographic context
   - Vary market maturity and business practices
   - Include cross-border or international elements

5. **Very Hard Negative - Hybrid Role Confusion**
   - Combine responsibilities from multiple distinct roles
   - Create plausible but incorrect role combinations
   - Mix strategic and tactical responsibilities inappropriately
   - Include overlapping but different skill requirements

### Output Format:
**Positive Description**:
The [Job Role] ...

**Negative Description**:
The [Job Role] ...
"""



In [36]:
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import GenerateSentencePair

with Pipeline(name="generate") as pipeline:
    load_dataset = LoadDataFromHub(
        num_examples=10,  # Limit to 10 examples for demo - increase for production datasets
        use_cache=False,  # Disable caching to ensure fresh data generation each run
        output_mappings={"Job Role Description": "anchor"},  # Map original column to 'anchor' for triplet generation
    )
    generate_retrieval_pairs_easy = GenerateSentencePair(
        name="easy_triplets_paraphrase",
        triplet=True,  # Generate anchor-positive-negative triplets for embedding training
        hard_negative=False,  # Use easier negatives rather than hard negatives
        action="paraphrase",  # Focus on paraphrasing for positive examples
        llm=llm,  # Use the LLM configured above (local Qwen or OpenAI)
        input_batch_size=2,  # Process 10 examples at once for efficiency
        #context=context,  # Provide the context instructions for generation quality
        context=context,  # Provide the context instructions for generation quality
    )
    generate_retrieval_pairs_hard = GenerateSentencePair(
        name="hard_triplets_paraphrase",
        triplet=True,  
        hard_negative=True,  
        action="paraphrase",  
        llm=llm,  
        input_batch_size=2,  
        #context=context,  
        context=context,  
    )

    load_dataset.connect(generate_retrieval_pairs_easy, generate_retrieval_pairs_hard)

In [37]:
output_avg_token_length = token_avg_length_rounded*2
output_max_token_length = max_token_length*2

print("Output Token Length:", output_avg_token_length)
print("Output Max Token Length:", output_max_token_length)

Output Token Length: 324
Output Max Token Length: 786


In [38]:
distiset = pipeline.run(
    use_cache=False,
    parameters={
        load_dataset.name: {
            "repo_id": "dnth/ssf-dataset",
            "split": "train",
        },
        "easy_triplets_paraphrase": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": output_max_token_length}}
             #"llm": {"generation_kwargs": {"temperature": 0.6}}
        },
        "hard_triplets_paraphrase": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": output_max_token_length}}
             #"llm": {"generation_kwargs": {"temperature": 0.6}}
        },
    }
)

[09/09/25 17:44:01] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=67493;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=684739;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/home/frank123/.cache/distilabel/pipelines/generate/5452575778c9f9fcb171             
                             f296279096b5c1119436/executions/e01275c71921ca282e67581ddc9a428038cbeed2/             
                             data/steps_outputs'                                                                   

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=216340;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=940478;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_hub_0'                                                        
                                - 🔄 'easy_triplets_paraphrase'                                                    
                                - 🔄 'hard_triplets_paraphrase'                                                    

                    INFO     ['distilabel.pipeline'] ⏳ Waiting for all the steps of stage 0 to        ]8;id=643256;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=237199;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1382\1382]8;;\
                             load...                                                                               

[09/09/25 17:44:04] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 2/3                 ]8;id=524543;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=371930;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 0/1                                               
                              * 'easy_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_paraphrase' replicas: 1/1                                           

[09/09/25 17:44:06] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 3/3                 ]8;id=623663;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=898509;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 1/1                                               
                              * 'easy_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_paraphrase' replicas: 1/1                                           

...

                    INFO     ['distilabel.step.easy_triplets_paraphrase'] 🏁 Finished running   ]8;id=755959;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=736419;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             step 'easy_triplets_paraphrase' (replica ID: 0)                                       

[09/09/25 17:44:37] INFO     ['distilabel.step.hard_triplets_paraphrase'] 📨 Step               ]8;id=366295;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=550232;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py#290\290]8;;\
                             'hard_triplets_paraphrase' sending batch 4 to output queue                            

                    INFO     ['distilabel.step.hard_triplets_paraphrase'] 🏁 Finished running   ]8;id=920464;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=158441;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             step 'hard_triplets_paraphrase' (replica ID: 0)                                       

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [39]:
distiset

Distiset({
    easy_triplets_paraphrase: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 10
        })
    })
    hard_triplets_paraphrase: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 10
        })
    })
})

In [40]:
distiset["hard_triplets_paraphrase"]["train"][-1]

{'Sector': 'Accountancy',
 'Track': 'Enterprise Risk Management',
 'Job Role': 'Enterprise Risk Management Associate / Enterprise Risk Management Executive',
 'anchor': "The Enterprise Risk Management Associate/Enterprise Risk Management Executive is responsible for supporting the implementation of enterprise risk management (ERM) activities, as well as policy and process maintenance. He/She gathers information, monitors and flags issues within ERM systems. He assists in preparing documents and reports for management review. He monitors adherence to risk policy and guidelines, supporting overall communication and risk reporting mechanisms. He also supports identification of resolution activities after high-risk incidents. The Enterprise Risk Management Associate/Enterprise Risk Management Executive is inquisitive, adaptable, a quick learner and is able to execute work independently. He is highly motivated, takes initiative and able to deliver outcomes as required. He is also analytical

In [41]:
# hard_triplets_semantic_df = distiset["hard_triplets_paraphrase"]["train"].to_pandas()
hard_triplets_semantic_df = distiset["easy_triplets_paraphrase"]["train"].to_pandas()
hard_triplets_semantic_df

,Sector,Track,Job Role,anchor,Performance Expectation,positive,negative,distilabel_metadata,model_name
0,Accountancy,Assurance,Audit Associate / Audit Assistant Associate,The Audit Associate/Audit Assistant Associate ...,In accordance with: Singapore Standards on Aud...,The Audit Associate is responsible for executi...,Description (Easy Negative - Different Functio...,{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
1,Accountancy,Assurance,Audit Manager,The Audit Senior Manager/Audit Manager manages...,In accordance with: Singapore Standards on Aud...,The Audit Senior Manager oversees a range of c...,"(Easy Negative - Different Function, Same Indu...",{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
2,Accountancy,Assurance,Audit Partner / Audit Director,The Audit Partner/Audit Director is a transfor...,In accordance with: Singapore Standards on Aud...,The Audit Partner is a visionary leader who gu...,"(Medium Negative - Same Industry, Different Se...",{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
3,Accountancy,Assurance,Audit Senior,The Audit Senior is expected to team lead vari...,In accordance with: Singapore Standards on Aud...,The Audit Senior is responsible for leading au...,"(Medium Negative - Same Industry, Different Se...",{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
4,Accountancy,Business Valuation,Business Valuation Associate / Business Valuat...,The Business Valuation Associate/Business Valu...,In accordance with the International Valuation...,The Business Valuation Executive plays a cruci...,"(Easy Negative - Different Function, Same Indu...",{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
5,Accountancy,Business Valuation,Business Valuation Manager,The Business Valuation Manager is second in ch...,In accordance with the International Valuation...,The Business Valuation Manager plays a pivotal...,"(Easy Negative - Different Function, Same Indu...",{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
6,Accountancy,Business Valuation,Business Valuation Partner / Business Valuatio...,The Business Valuation Partner/Business Valuat...,In accordance with the International Valuation...,The Business Valuation Director leads a dedica...,Description (Easy Negative - Different Functio...,{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
7,Accountancy,Business Valuation,Business Valuation Senior / Business Valuation...,The Business Valuation Senior/Business Valuati...,In accordance with the International Valuation...,The Business Valuation Senior Executive overse...,"(Medium Negative - Same Industry, Different Se...",{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
8,Accountancy,Enterprise Risk Management,Chief Risk Officer / Risk Partner / Head of Ri...,The Chief Risk Officer/Risk Partner/Head of Ri...,None,The Chief Risk Officer is responsible for over...,Description (Easy Negative - Different Functio...,{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini
9,Accountancy,Enterprise Risk Management,Enterprise Risk Management Associate / Enterpr...,The Enterprise Risk Management Associate/Enter...,None,The Enterprise Risk Management Associate is ta...,Description (Easy Negative - Different Functio...,{'raw_input_easy_triplets_paraphrase': [{'cont...,gpt-4o-mini


In [ ]:
#distiset.push_to_hub("frankwong2001/ssf-dataset-synthetic_test_2")
distiset.push_to_hub("frankwong2001/ssf-dataset_Full_synthetic_batch10")